# FDA Drug Label Parser

In [1]:
import requests
import xml.etree.ElementTree as ET
import json
import re

print("Done.")

Done.


In [2]:
DRUG_NAME = "levetiracetam"  # change this to any drug

search_url = f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls.json?drug_name={DRUG_NAME}"
r = requests.get(search_url, timeout=15)
results = r.json().get('data', [])

print(f"Found {len(results)} labels for '{DRUG_NAME}'")
for i, item in enumerate(results[:5]):
    print(f"  [{i}] set_id: {item['setid']}  title: {item.get('title','')[:80]}")

Found 100 labels for 'levetiracetam'
  [0] set_id: a05bbfa4-c34f-4134-8207-db79e4c70445  title: LEVETIRACETAM TABLET [REMEDYREPACK INC.]
  [1] set_id: f924d55d-d7ec-4448-b54e-3a7eaf5a1cab  title: LEVETIRACETAM SOLUTION [CARDINAL HEALTH 107, LLC]
  [2] set_id: 23073db5-3fc4-4051-a22d-e49a63e42647  title: LEVETIRACETAM TABLET [REMEDYREPACK INC.]
  [3] set_id: 74ccc9f1-b957-4b1b-9464-1b5e3318c916  title: LEVETIRACETAM SOLUTION [REMEDYREPACK INC.]
  [4] set_id: 94d042f7-93b9-43bd-9ace-cfc61903b4d7  title: LEVETIRACETAM SOLUTION [REMEDYREPACK INC.]


In [3]:
# Pick the first result (index 0) — change if you want a different one
set_id = results[0]['setid']

label_url = f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{set_id}.xml"
xml_data = requests.get(label_url, timeout=15).text

print(f"Fetched XML — {len(xml_data):,} characters")
print()
print(xml_data[:500])

Fetched XML — 270,424 characters

<?xml version="1.0" encoding="UTF-8"?><?xml-stylesheet href="https://www.accessdata.fda.gov/spl/stylesheet/spl.xsl" type="text/xsl"?>
<document xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="urn:hl7-org:v3" xsi:schemaLocation="urn:hl7-org:v3 https://www.accessdata.fda.gov/spl/schema/spl.xsd">
   <id root="541018df-7c70-8cc2-e063-6394a90ae0ff"/>
   <code code="34391-3" codeSystem="2.16.840.1.113883.6.1" displayName="HUMAN PRESCRIPTION DRUG LABEL"/>
   <title>
      <content styleCod


---
### Map the structure

In [4]:
NS = {'hl7': 'urn:hl7-org:v3'}
tree = ET.fromstring(xml_data)
sections = tree.findall('.//hl7:section', NS)

print(f"Total sections found: {len(sections)}")
print()
print(f"{'CODE':<15} {'DISPLAY NAME':<45} TITLE")
print("-" * 90)

for s in sections:
    code_el = s.find('hl7:code', NS)
    title_el = s.find('hl7:title', NS)
    code     = code_el.attrib.get('code', '?') if code_el is not None else '?'
    display  = code_el.attrib.get('displayName', '?') if code_el is not None else '?'
    title    = title_el.text if title_el is not None else '(no title)'
    print(f"{code:<15} {display:<45} {title}")

Total sections found: 56

CODE            DISPLAY NAME                                  TITLE
------------------------------------------------------------------------------------------
48780-1         SPL product data elements section             (no title)
43683-2         RECENT MAJOR CHANGES SECTION                  None
34067-9         INDICATIONS & USAGE SECTION                   1 INDICATIONS AND USAGE
42229-5         SPL UNCLASSIFIED SECTION                      1.1 Partial-Onset Seizures
42229-5         SPL UNCLASSIFIED SECTION                      1.2 Myoclonic Seizures in Patients with Juvenile Myoclonic Epilepsy
42229-5         SPL UNCLASSIFIED SECTION                      1.3 Primary Generalized Tonic-Clonic Seizures
34068-7         DOSAGE & ADMINISTRATION SECTION               2 DOSAGE AND ADMINISTRATION
42229-5         SPL UNCLASSIFIED SECTION                      2.1 Important Administration Instructions
42229-5         SPL UNCLASSIFIED SECTION                      2.2 Do

---
### Define which sections to keep vs skip

In [5]:
# Sections worth keeping for RAG — LOINC code → readable label
KEEP_SECTIONS = {
    '34067-9': 'Indications and Usage',
    '34068-7': 'Dosage and Administration',
    '34070-3': 'Contraindications',
    '43685-7': 'Warnings and Precautions',
    '34084-4': 'Adverse Reactions',
    '43684-0': 'Use in Specific Populations',
    '42228-7': 'Pregnancy',
    '34081-0': 'Pediatric Use',
    '34082-8': 'Geriatric Use',
    '42229-5': 'Subsection',
    '34088-5': 'Overdosage',
    '34090-1': 'Clinical Pharmacology',
    '43679-0': 'Mechanism of Action',
    '43682-4': 'Pharmacokinetics',
    '34092-7': 'Clinical Studies',
    '43683-2': 'Recent Major Changes',
}

# Sections to skip — metadata, packaging, patient-facing duplicates
SKIP_SECTIONS = {
    '48780-1',  # product data elements (NDC codes)
    '51945-4',  # package label / display panel
    '42231-1',  # medguide (simplified patient duplicate)
    '34069-5',  # how supplied / storage
    '34076-0',  # patient counseling info
    '34083-6',  # carcinogenesis (nonclinical)
    '43680-8',  # nonclinical toxicology
}

print(f"Keeping : {len(KEEP_SECTIONS)} section types")
print(f"Skipping: {len(SKIP_SECTIONS)} section types")

Keeping : 16 section types
Skipping: 7 section types


---
### Cleaning the text and normalize everything

In [6]:
def clean_text(element):
    """Recursively extract plain text from XML, stripping all tags."""
    parts = []
    if element.text:
        parts.append(element.text.strip())
    for child in element:
        parts.append(clean_text(child))
        if child.tail:
            parts.append(child.tail.strip())
    return ' '.join(p for p in parts if p)


def normalize(text):
    """Collapse whitespace and clean punctuation artifacts."""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\s([.,;:])', r'\1', text)
    return text.strip()


# Quick test
sample = tree.findall('.//hl7:section', NS)[2]
sample_text_el = sample.find('hl7:text', NS)
if sample_text_el is not None:
    print(normalize(clean_text(sample_text_el))[:300])
else:
    print("(no text in this section — try another index)")

---
### parsing into chunks

In [7]:
def parse_label(xml_data, drug_name):
    """
    Walk the SPL XML and return a flat list of chunks.
    Each chunk: drug, section, subsection, title, text, loinc_code.
    """
    tree = ET.fromstring(xml_data)
    chunks = []

    top_sections = tree.findall(
        './/hl7:component/hl7:structuredBody/hl7:component/hl7:section', NS
    )

    for top in top_sections:
        code_el = top.find('hl7:code', NS)
        if code_el is None:
            continue
        top_code = code_el.attrib.get('code', '')

        if top_code in SKIP_SECTIONS:
            continue

        top_label = KEEP_SECTIONS.get(
            top_code, code_el.attrib.get('displayName', 'Unknown')
        )
        top_title_el = top.find('hl7:title', NS)
        top_title = normalize(clean_text(top_title_el)) if top_title_el is not None else top_label

        # Direct text of the parent section
        top_text_el = top.find('hl7:text', NS)
        if top_text_el is not None:
            top_text = normalize(clean_text(top_text_el))
            if len(top_text) > 40:
                chunks.append({
                    'drug'       : drug_name,
                    'section'    : top_label,
                    'subsection' : None,
                    'title'      : top_title,
                    'text'       : top_text,
                    'loinc_code' : top_code,
                })

        # Subsections
        for sub in top.findall('.//hl7:component/hl7:section', NS):
            sub_title_el = sub.find('hl7:title', NS)
            sub_text_el  = sub.find('hl7:text', NS)
            sub_title = normalize(clean_text(sub_title_el)) if sub_title_el is not None else None
            sub_text  = normalize(clean_text(sub_text_el))  if sub_text_el  is not None else None

            if sub_text and len(sub_text) > 40:
                chunks.append({
                    'drug'       : drug_name,
                    'section'    : top_label,
                    'subsection' : sub_title,
                    'title'      : sub_title or top_title,
                    'text'       : sub_text,
                    'loinc_code' : top_code,
                })

    return chunks


chunks = parse_label(xml_data, DRUG_NAME)
print(f"Total chunks: {len(chunks)}")

Total chunks: 41


---
### Inspect the chunks

In [8]:
# Summary table — all chunks
print(f"{'#':<4} {'SECTION':<35} {'SUBSECTION':<45} CHARS")
print("-" * 100)
for i, c in enumerate(chunks):
    sub = (c['subsection'] or '')[:43]
    print(f"{i:<4} {c['section']:<35} {sub:<45} {len(c['text'])}")

#    SECTION                             SUBSECTION                                    CHARS
----------------------------------------------------------------------------------------------------
0    Indications and Usage               1.1 Partial-Onset Seizures                    108
1    Indications and Usage               1.2 Myoclonic Seizures in Patients with Juv   160
2    Indications and Usage               1.3 Primary Generalized Tonic-Clonic Seizur   186
3    Dosage and Administration           2.1 Important Administration Instructions     629
4    Dosage and Administration           2.2 Dosing for Partial-Onset Seizures         2522
5    Dosage and Administration           2.3 Dosing for Myoclonic Seizures in Patien   259
6    Dosage and Administration           2.4 Dosing for Primary Generalized Tonic-Cl   886
7    Dosage and Administration           2.5 Dosage Adjustments in Adult Patients wi   1108
8    Dosage and Administration           2.6 Discontinuation of Levetiraceta

In [9]:
# Read a specific chunk in full — change the index
CHUNK_INDEX = 0

c = chunks[CHUNK_INDEX]
print(f"Drug      : {c['drug']}")
print(f"Section   : {c['section']}")
print(f"Subsection: {c['subsection']}")
print(f"LOINC     : {c['loinc_code']}")
print(f"Chars     : {len(c['text'])}")
print()
print(c['text'])

Drug      : levetiracetam
Section   : Indications and Usage
Subsection: 1.1 Partial-Onset Seizures
LOINC     : 34067-9
Chars     : 108

Levetiracetam is indicated for the treatment of partial-onset seizures in patients 1 month of age and older.


---
### Format for embedding

In [10]:
def format_for_embedding(chunk):
    """
    Prepend metadata so the embedding captures drug + section context,
    not just the raw clinical text.
    """
    lines = [f"Drug: {chunk['drug']}"]
    lines.append(f"Section: {chunk['section']}")
    if chunk['subsection']:
        lines.append(f"Subsection: {chunk['subsection']}")
    lines.append(f"\n{chunk['text']}")
    return '\n'.join(lines)


# Preview first 3 embedding strings
for i, c in enumerate(chunks[:3]):
    print(f"{'='*60}")
    print(f"[Chunk {i}]")
    print(format_for_embedding(c)[:400])
    print()

[Chunk 0]
Drug: levetiracetam
Section: Indications and Usage
Subsection: 1.1 Partial-Onset Seizures

Levetiracetam is indicated for the treatment of partial-onset seizures in patients 1 month of age and older.

[Chunk 1]
Drug: levetiracetam
Section: Indications and Usage
Subsection: 1.2 Myoclonic Seizures in Patients with Juvenile Myoclonic Epilepsy

Levetiracetam is indicated as adjunctive therapy for the treatment of myoclonic seizures in patients 12 years of age and older with juvenile myoclonic epilepsy.

[Chunk 2]
Drug: levetiracetam
Section: Indications and Usage
Subsection: 1.3 Primary Generalized Tonic-Clonic Seizures

Levetiracetam is indicated as adjunctive therapy for the treatment of primary generalized tonic-clonic seizures in patients 6 years of age and older with idiopathic generalized epilepsy.



In [11]:
output_file = f"{DRUG_NAME}_chunks.json"

with open(output_file, 'w') as f:
    json.dump(chunks, f, indent=2)

print(f"Saved {len(chunks)} chunks to {output_file}")

# Preview the file structure
with open(output_file) as f:
    preview = json.load(f)
print()
print(json.dumps(preview[0], indent=2))

Saved 41 chunks to levetiracetam_chunks.json

{
  "drug": "levetiracetam",
  "section": "Indications and Usage",
  "subsection": "1.1 Partial-Onset Seizures",
  "title": "1.1 Partial-Onset Seizures",
  "text": "Levetiracetam is indicated for the treatment of partial-onset seizures in patients 1 month of age and older.",
  "loinc_code": "34067-9"
}


---
### Run on different type of drugs

In [13]:
# Add as many UCB-relevant drugs as you want
DRUG_LIST = [
    # Epilepsy — core portfolio
    "levetiracetam",
    "brivaracetam",
    
    # Epilepsy — related AEDs for competitive context
    "lacosamide",
    "zonisamide",
    "topiramate",
    "lamotrigine",
    "valproate",
    "carbamazepine",
    "oxcarbazepine",
    "perampanel",
    
    # # Immunology / neurology — UCB biologics
    # "certolizumab pegol",
    # "romosozumab",
    # "zilucoplan",
    # "rozanolixizumab",
    
    # # Parkinson's — UCB pipeline
    # "apomorphine",
]

all_chunks = []

for drug in DRUG_LIST:
    try:
        print(f"Fetching: {drug}...")
        xml = requests.get(
            f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls.json?drug_name={drug}",
            timeout=15
        ).json()
        results = xml.get('data', [])
        if not results:
            print(f"  No results found for {drug}")
            continue
        sid = results[0]['setid']
        xml_data_drug = requests.get(
            f"https://dailymed.nlm.nih.gov/dailymed/services/v2/spls/{sid}.xml",
            timeout=15
        ).text
        drug_chunks = parse_label(xml_data_drug, drug)
        all_chunks.extend(drug_chunks)
        print(f"  {len(drug_chunks)} chunks extracted")
    except Exception as e:
        print(f"  Error: {e}")

print(f"\nTotal across all drugs: {len(all_chunks)} chunks")

with open("all_drugs_chunks.json", "w") as f:
    json.dump(all_chunks, f, indent=2)
print("Saved to all_drugs_chunks.json")

Fetching: levetiracetam...
  41 chunks extracted
Fetching: brivaracetam...
  38 chunks extracted
Fetching: lacosamide...
  39 chunks extracted
Fetching: zonisamide...
  18 chunks extracted
Fetching: topiramate...
  51 chunks extracted
Fetching: lamotrigine...
  44 chunks extracted
Fetching: valproate...
  115 chunks extracted
Fetching: carbamazepine...
  32 chunks extracted
Fetching: oxcarbazepine...
  45 chunks extracted
Fetching: perampanel...
  84 chunks extracted

Total across all drugs: 507 chunks
Saved to all_drugs_chunks.json
